In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from collections import Counter
import emoji

In [ ]:
file0 = 'Apna DARF'+'.docx'

In [ ]:
file = open(file0, 'r', encoding = 'utf-8')
data = file.read()

pattern = '\d{1,2}/\d{1,2}/\d{1,4},\s\d{1,2}:\d{1,2}\s-\s'

messages = re.split(pattern, data)[1:]
dates = re.findall(pattern, data)

df = pd.DataFrame({ 'User Message': messages,
                   'M_Date': dates
                  })
df['M_Date'] = pd.to_datetime(df['M_Date'], format='%d/%m/%y, %H:%M - ')
df.rename(columns = {'M_Date': 'Date'}, inplace=True)

users = []
message1 = []

for msg in df['User Message']:
    splitting = re.split('([\w\W]+?):\s', msg)
    if splitting[1:]:
        users.append(splitting[1])
        message1.append(splitting[2])
    else:
        users.append('group_notification')
        message1.append(splitting[0])


df['User'] = users
df['Message'] = message1
df.drop(columns=['User Message'], inplace=True)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month_name()
df['Day']  = df['Date'].dt.day
df['Hour'] = df['Date'].dt.hour
df['Minute'] = df['Date'].dt.minute
    
df.head()

In [ ]:
f = open('stop words.txt', 'r')
stop_words = f.read()
temp = df[df['User'] != 'group notification']
temp = temp[temp['Message'] != '<Media omitted>\n']

words = []

for _ in temp['Message']:
    for i in _.lower().split():
        if i not in stop_words.split():
            words.append(i)
x = pd.DataFrame(Counter(words).most_common(20))
x.rename(columns={0:'words', 1:'Repeated'}, inplace=True)

x

In [ ]:
df['month_number'] = df['Date'].dt.month
duration = df.groupby(['Year','month_number', 'Month']).count()['Message'].reset_index()
month_year = []
for _ in range(duration.shape[0]):
    month_year.append(duration['Month'][_] + "-" + str(duration['Year'][_]))
duration['Month-Year'] = month_year
duration['Month-Year'] 

In [ ]:
def stats(select_user,df):
    fig, axis = plt.subplots(5, figsize=(30, 30))
    if select_user == 'ALL':
        total_messages = df.shape[0]
        media_messages = df[df['Message'] == '<Media omitted>\n'].shape[0]
        
        m_fig = x.plot(kind = 'barh',
              x = 'words',
              y = 'Repeated', ax =axis[0], figsize=(20, 20))
        
        axis[0].title.set_text("Most Common Words")
        
        number_of_words = []
        for message in df['Message']:
            number_of_words.extend(message.split())
        
        print(f"Total Messages = {total_messages}") 
        print(f"Total Number of Words = {(len(number_of_words))}")
        print(f"Total Media Messages = {media_messages}")
    
    else:
        new_df = (df[df['User'] == select_user])
        total_messages = new_df.shape[0]

        number_of_words = []
        for message in df['Message']:
            number_of_words.extend(message.split())
        
        m_fig = x.plot(kind = 'barh',
          x = 'words',
          y = 'Repeated')
        plt.title("Most Common Words")
            
        print(f"Total Messages = {total_messages}") 
        print(f"Total Number of Words = {(len(number_of_words))}")

    i = df['User'].value_counts().head()
    name = i.index
    count = i.values
    axis[1].title.set_text("Number of messages by individuals")
    axis[1].bar(name, count)
    
    duration.plot(kind = 'line',
      x = 'Month-Year',
      y = 'Message',ax = axis[2])
    axis[2].set_xlabel('Month-Year')
    axis[2].set_ylabel('Number of messaages')
    axis[2].title.set_text('Month vs Messages')
    
    df['date'] = df['Date'].dt.date
    daily_messages = df.groupby(['date']).count()['Message'].reset_index()
    daily_messages.plot(kind = 'line',
          x = 'date',
          y = 'Message', ax=axis[3])
    axis[3].set_xlabel('Days')
    axis[3].set_ylabel('Number of Daily Messages')
    axis[3].set_title('Days vs Messages')  
    
    df['day_name'] = df['Date'].dt.day_name()
    day_name = df['day_name'].value_counts().reset_index()
    day_name.rename(columns={'index':'Day Name', 'day_name':'Number of Messages'}, inplace=True)
    day_name.plot(kind = 'barh',
      x = 'Day Name',
      y = 'Number of Messages', ax = axis[4])
    axis[4].set_xlabel('Days')
    axis[4].set_ylabel('Number of Daily Messages')
    axis[4].set_title('Days vs Messages')
    
    plt.tight_layout()

In [ ]:
select_user = str(input())

In [ ]:
stats(select_user,df)